# Hello world example with Legate tasks

## What you need to know for this tutorial
1. A basic knowledge of Jax and numpy
2. Familiarity with Jax meshes and devices

## Initialize Legate-Jax
The first step in a Legate-Jax program is to initialize the environment. Before any other imports, `jax_plugins.legate.init` should be called with the appropriate arguments for configuring Legate. This does violate the common Python style rule of top-level imports coming before all code. The parameters for Legate must be configured before importing any Jax functions to ensure that backend clients are initialized properly.

In [1]:
from jax_plugins.legate import init
init(cpus=2, dump="dump")

## Import Jax packages

Once Legate-Jax has been initialized, the standard set of Jax imports can be done. We also import the special Legate-Jax functions.

In [2]:
import jax
import jax.numpy as jnp
import numpy as np
from legate.jax import task, mjit

## Jax example with two functions

We show a Jax example executing two functions. We `jax.jit` the composite function into a single callable and pass in an example input. As expected, the result is an array of 1's (within rounding error).

In [3]:
def f(x):
    return jnp.cos(x), jnp.sin(x)

def g(x,y):
    return x*x + y*y

@jax.jit
def c(x):
    return g(*f(x))

batch = jnp.arange(8)
result = c(batch)
result

Platform 'legate' is experimental and not all JAX functionality may be correctly supported!
I0000 00:00:1724647834.552920 1932354 cpu_client.cc:466] TfrtCpuClient created.


Array([1.        , 0.99999994, 0.99999994, 1.        , 1.        ,
       1.        , 0.99999994, 1.        ], dtype=float32)

The computation is not sharded so the computation and result is placed on device 0 by default.

In [4]:
result.devices()

{CpuDevice(id=0)}

## Jax example with SPMD sharding

We can extend the example by passing in a sharded array and producing a sharded array. The result will now be distributed across multiple devices. We define the sharding of the array using partition specs and named shardings, which in this case defines a 1-D array sharding across the axis "x" of size 2. For more details, consult the [Jax documentation on sharding](https://jax.readthedocs.io/en/latest/notebooks/Distributed_arrays_and_automatic_parallelization.html).  

In [5]:
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P
from jax.experimental.mesh_utils import create_device_mesh

mesh = Mesh(np.array(jax.devices()), ("x",))
sharding = NamedSharding(mesh, P("x"))
batch = jax.jit(jnp.arange, out_shardings=sharding, static_argnums=0)(8)
result = c(batch)
result.devices()

{CpuDevice(id=0), CpuDevice(id=1)}

## Legate-Jax example with MPMD execution

Jax generally limits computations to SPMD with all devices executing the same computation.  MPMD patterns where different operations are executed on difference devices can be difficult or impossible to express with native Jax idioms.  The Legate runtime adds this support through the @task decorator on functions (and Flax modules in later examples). Instead of running `f` and `g` functions across both devices, we run `f` on device 0 and `g` on device 1.  The input `batch` is placed on device 0, but the output is placed on device 1.

In [6]:
all_devices = np.array(jax.devices())

legate_f = task(f, devices=all_devices[0:1])
legate_g = task(g, devices=all_devices[1:2])

def legate_c(x):
    return legate_g(*legate_f(x))
legate_c = mjit(legate_c, devices=all_devices)

batch = jnp.arange(8)
result = legate_c(batch)
f"batch={batch.devices()}, result={result.devices()}"

W0000 00:00:1724647835.695261 1932354 hlo_module_config.h:186] Warning: Using auto_spmd_partitioning. It is experimental and may contain bugs!
I0000 00:00:1724647835.695277 1932354 hlo_module_config.h:188] Overwriting use_spmd_partitioning to true, because use_auto_spmd_partitioning is true.
W0000 00:00:1724647835.696606 1932354 hlo_module_config.h:186] Warning: Using auto_spmd_partitioning. It is experimental and may contain bugs!
I0000 00:00:1724647835.696613 1932354 hlo_module_config.h:188] Overwriting use_spmd_partitioning to true, because use_auto_spmd_partitioning is true.
W0000 00:00:1724647835.696883 1932354 hlo_module_config.h:186] Warning: Using auto_spmd_partitioning. It is experimental and may contain bugs!
I0000 00:00:1724647835.696889 1932354 hlo_module_config.h:188] Overwriting use_spmd_partitioning to true, because use_auto_spmd_partitioning is true.


'batch={CpuDevice(id=0)}, result={CpuDevice(id=1)}'